In [11]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("F1_Modelado")
    .enableHiveSupport()
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RFN_PATH = "ort/RFN"
MDL_PATH = "ort/MDL"

def leer(nombre):
    return spark.read.parquet(f"{RFN_PATH}/{nombre}")

def guardar_mdl(df, nombre):
    df.write.mode("overwrite").parquet(f"{MDL_PATH}/{nombre}")
    print(f"[MDL] {nombre:28s} -> {df.count()} filas, {len(df.columns)} cols")
    return df

In [12]:
# --- Dimensiones ---
dim_driver = leer("drivers")
dim_constructor = leer("constructors")
dim_circuit = leer("circuits")
dim_status = leer("status")
dim_race = leer("races").select(
    "raceId", "season", "round", "raceName", "date", "circuitId")

In [13]:
# Confirmación de tabla results, nos aseguramos de que no existan status (strings) sin su correspondiente Id

res = leer("results")

sin_match = (res.join(dim_status, res["status"] == dim_status["status"], "left")
                .filter(dim_status["statusId"].isNull())
                .select(res["status"]).distinct())

print("Textos de results.status sin correspondencia en dim_status:", sin_match.count())
sin_match.show(50, truncate=False)

Textos de results.status sin correspondencia en dim_status: 0
+------+
|status|
+------+
+------+



In [14]:
# --- Hechos ---
fact_results = (res.join(dim_status.select("statusId", "status"), on="status", how="left")
    .select(
        "resultId", "raceId", "driverId", "constructorId", "statusId",
        "grid", "position", "positionText", "points", "laps",
    ))

fact_cs = leer("constructor_standings").select(
    "standingId", "constructorId", "season", "totalRounds",
    "points", "position", "positionText", "wins")

In [15]:
# --- Guardado (todo junto, al final) ---
guardar_mdl(dim_driver, "dim_driver")
guardar_mdl(dim_constructor, "dim_constructor")
guardar_mdl(dim_circuit, "dim_circuit")
guardar_mdl(dim_status, "dim_status")
guardar_mdl(dim_race, "dim_race")
guardar_mdl(fact_results, "fact_results")
guardar_mdl(fact_cs, "fact_constructor_standings")

[MDL] dim_driver                   -> 881 filas, 7 cols
[MDL] dim_constructor              -> 214 filas, 3 cols
[MDL] dim_circuit                  -> 78 filas, 6 cols
[MDL] dim_status                   -> 136 filas, 2 cols
[MDL] dim_race                     -> 503 filas, 6 cols


[MDL] fact_results                 -> 10550 filas, 10 cols
[MDL] fact_constructor_standings   -> 276 filas, 8 cols


DataFrame[standingId: string, constructorId: string, season: int, totalRounds: int, points: double, position: int, positionText: string, wins: int]

In [16]:
assert dim_status.count() == dim_status.select("status").distinct().count()